# Análisis exploratorio y consultas de negocio

Consultas SQL sobre la capa Gold, respondiendo preguntas de negocio del challenge.

In [0]:
from databricks.sdk.runtime import spark, dbutils, display

CATALOGO = "workspace"
SCHEMA = "default"

## 1. Clientes que operaron consistentemente por encima del precio de mercado

"Consistentemente" se interpreta como: en más del 70% de sus compras, el precio
operado superó al precio de mercado. Solo se consideran clientes con al menos
5 compras con cotización disponible, para evitar que 1-2 operaciones aisladas
distorsionen el resultado.

In [0]:
spark.sql(f"""
    SELECT
        c.id_cliente,
        i.simbolo_base,
        i.tipo_instrumento,
        COUNT(*) as compras_con_cotizacion,
        SUM(CASE WHEN f.precio_operado > f.precio_mercado THEN 1 ELSE 0 END) as compras_sobre_mercado,
        ROUND(
            SUM(CASE WHEN f.precio_operado > f.precio_mercado THEN 1 ELSE 0 END) / COUNT(*) * 100, 1
        ) as pct_sobre_mercado
    FROM {CATALOGO}.{SCHEMA}.fact_transacciones f
    JOIN {CATALOGO}.{SCHEMA}.dim_cliente c ON f.sk_cliente = c.sk_cliente
    JOIN {CATALOGO}.{SCHEMA}.dim_instrumento i ON f.sk_instrumento = i.sk_instrumento
    WHERE f.tipoTran = 'Compra' AND f.precio_mercado IS NOT NULL
    GROUP BY c.id_cliente, i.simbolo_base, i.tipo_instrumento
    HAVING COUNT(*) >= 5
    ORDER BY pct_sobre_mercado DESC, compras_con_cotizacion DESC
    LIMIT 20
""").show(20, truncate=False)

+-----------+------------+----------------+----------------------+---------------------+-----------------+
|id_cliente |simbolo_base|tipo_instrumento|compras_con_cotizacion|compras_sobre_mercado|pct_sobre_mercado|
+-----------+------------+----------------+----------------------+---------------------+-----------------+
|CLI6B178993|EDN         |Acción local    |5                     |5                    |100.0            |
|CLI5FC09479|GGAL        |Acción local    |5                     |5                    |100.0            |
|CLI5F2351E6|HUT         |CEDEAR          |5                     |4                    |80.0             |
|CLIDAB0BA0E|COME        |Acción local    |5                     |4                    |80.0             |
|CLIF1269A3E|COME        |Acción local    |8                     |6                    |75.0             |
|CLI6F6E6E2E|COME        |Acción local    |14                    |10                   |71.4             |
|CLIE3F505BE|COME        |Acción loca

## 2. Instrumentos con mayor desvío promedio entre precio operado y precio de mercado

In [0]:
spark.sql(f"""
    SELECT
        i.simbolo_base,
        i.tipo_instrumento,
        COUNT(*) as transacciones_con_cotizacion,
        ROUND(AVG(ABS(f.desvio_pct)) * 100, 2) as desvio_promedio_pct,
        ROUND(AVG(f.desvio_pct) * 100, 2) as desvio_promedio_con_signo_pct
    FROM {CATALOGO}.{SCHEMA}.fact_transacciones f
    JOIN {CATALOGO}.{SCHEMA}.dim_instrumento i ON f.sk_instrumento = i.sk_instrumento
    WHERE f.desvio_pct IS NOT NULL
    GROUP BY i.simbolo_base, i.tipo_instrumento
    HAVING COUNT(*) >= 5
    ORDER BY desvio_promedio_pct DESC
    LIMIT 20
""").show(20, truncate=False)

+------------+----------------+----------------------------+-------------------+-----------------------------+
|simbolo_base|tipo_instrumento|transacciones_con_cotizacion|desvio_promedio_pct|desvio_promedio_con_signo_pct|
+------------+----------------+----------------------------+-------------------+-----------------------------+
|BB          |CEDEAR          |142                         |180.86             |180.70                       |
|AMD         |CEDEAR          |49                          |99.93              |-99.93                       |
|JD          |CEDEAR          |16                          |99.93              |-99.93                       |
|YPFD        |Acción local    |170                         |99.93              |-99.93                       |
|BBD         |CEDEAR          |17                          |99.93              |-99.93                       |
|MCD         |CEDEAR          |6                           |99.93              |-99.93                       |
|

## 3. Evolución de la proporción de operaciones por canal, mes a mes

In [0]:
spark.sql(f"""
    WITH totales_mes AS (
        SELECT d.anio, d.mes, COUNT(*) as total_mes
        FROM {CATALOGO}.{SCHEMA}.fact_transacciones f
        JOIN {CATALOGO}.{SCHEMA}.dim_fecha d ON f.sk_fecha = d.sk_fecha
        GROUP BY d.anio, d.mes
    )
    SELECT
        d.anio,
        d.mes,
        d.nombre_mes,
        ca.origen,
        COUNT(*) as operaciones,
        ROUND(COUNT(*) / t.total_mes * 100, 1) as pct_del_mes
    FROM {CATALOGO}.{SCHEMA}.fact_transacciones f
    JOIN {CATALOGO}.{SCHEMA}.dim_fecha d ON f.sk_fecha = d.sk_fecha
    JOIN {CATALOGO}.{SCHEMA}.dim_canal ca ON f.sk_canal = ca.sk_canal
    JOIN totales_mes t ON d.anio = t.anio AND d.mes = t.mes
    GROUP BY d.anio, d.mes, d.nombre_mes, ca.origen, t.total_mes
    ORDER BY d.anio, d.mes, pct_del_mes DESC
""").show(30, truncate=False)

+----+---+----------+--------------------+-----------+-----------+
|anio|mes|nombre_mes|origen              |operaciones|pct_del_mes|
+----+---+----------+--------------------+-----------+-----------+
|2026|1  |January   |App Mobile          |27734      |67.8       |
|2026|1  |January   |Sitio Web Desktop   |11522      |28.2       |
|2026|1  |January   |Sitio Web Responsive|1037       |2.5        |
|2026|1  |January   |API                 |468        |1.1        |
|2026|1  |January   |IOLnet              |118        |0.3        |
|2026|2  |February  |App Mobile          |19953      |65.3       |
|2026|2  |February  |Sitio Web Desktop   |9081       |29.7       |
|2026|2  |February  |Sitio Web Responsive|853        |2.8        |
|2026|2  |February  |API                 |564        |1.8        |
|2026|2  |February  |IOLnet              |93         |0.3        |
|2026|3  |March     |App Mobile          |10532      |65.5       |
|2026|3  |March     |Sitio Web Desktop   |4651       |28.9    

## 4. Retención de clientes: qué % de clientes de enero volvió a operar en febrero y marzo

In [0]:
spark.sql(f"""
    WITH clientes_por_mes AS (
        SELECT DISTINCT f.sk_cliente, d.mes
        FROM {CATALOGO}.{SCHEMA}.fact_transacciones f
        JOIN {CATALOGO}.{SCHEMA}.dim_fecha d ON f.sk_fecha = d.sk_fecha
    ),
    clientes_enero AS (
        SELECT sk_cliente FROM clientes_por_mes WHERE mes = 1
    )
    SELECT
        (SELECT COUNT(*) FROM clientes_enero) as clientes_enero,
        COUNT(DISTINCT CASE WHEN cm.mes = 2 THEN cm.sk_cliente END) as volvieron_febrero,
        COUNT(DISTINCT CASE WHEN cm.mes = 3 THEN cm.sk_cliente END) as volvieron_marzo,
        ROUND(
            COUNT(DISTINCT CASE WHEN cm.mes = 2 THEN cm.sk_cliente END) /
            (SELECT COUNT(*) FROM clientes_enero) * 100, 1
        ) as pct_retencion_febrero,
        ROUND(
            COUNT(DISTINCT CASE WHEN cm.mes = 3 THEN cm.sk_cliente END) /
            (SELECT COUNT(*) FROM clientes_enero) * 100, 1
        ) as pct_retencion_marzo
    FROM clientes_por_mes cm
    WHERE cm.sk_cliente IN (SELECT sk_cliente FROM clientes_enero)
""").show(truncate=False)

+--------------+-----------------+---------------+---------------------+-------------------+
|clientes_enero|volvieron_febrero|volvieron_marzo|pct_retencion_febrero|pct_retencion_marzo|
+--------------+-----------------+---------------+---------------------+-------------------+
|28125         |5497             |3177           |19.5                 |11.3               |
+--------------+-----------------+---------------+---------------------+-------------------+



## 5. Correlación entre canal de origen y probabilidad de comprar por encima del mercado

In [0]:
spark.sql(f"""
    SELECT
        ca.origen,
        COUNT(*) as compras_con_cotizacion,
        SUM(CASE WHEN f.precio_operado > f.precio_mercado THEN 1 ELSE 0 END) as compras_sobre_mercado,
        ROUND(
            SUM(CASE WHEN f.precio_operado > f.precio_mercado THEN 1 ELSE 0 END) / COUNT(*) * 100, 1
        ) as pct_compras_sobre_mercado
    FROM {CATALOGO}.{SCHEMA}.fact_transacciones f
    JOIN {CATALOGO}.{SCHEMA}.dim_canal ca ON f.sk_canal = ca.sk_canal
    WHERE f.tipoTran = 'Compra' AND f.precio_mercado IS NOT NULL
    GROUP BY ca.origen
    ORDER BY pct_compras_sobre_mercado DESC
""").show(truncate=False)

+--------------------+----------------------+---------------------+-------------------------+
|origen              |compras_con_cotizacion|compras_sobre_mercado|pct_compras_sobre_mercado|
+--------------------+----------------------+---------------------+-------------------------+
|Sitio Web Responsive|944                   |400                  |42.4                     |
|API                 |398                   |166                  |41.7                     |
|Sitio Web Desktop   |10485                 |3948                 |37.7                     |
|App Mobile          |26597                 |9094                 |34.2                     |
|IOLnet              |41                    |2                    |4.9                      |
+--------------------+----------------------+---------------------+-------------------------+



## 6. Pregunta propia: segmentación de clientes por actividad

¿Cómo se distribuyen los clientes según su nivel de actividad (cantidad de
operaciones en el período), y qué porcentaje del volumen total de transacciones
concentra cada segmento? Relevante para priorizar a qué clientes dirigir
comunicación o soporte comercial.

In [0]:
spark.sql(f"""
    WITH actividad_cliente AS (
        SELECT
            f.sk_cliente,
            COUNT(*) as total_operaciones
        FROM {CATALOGO}.{SCHEMA}.fact_transacciones f
        GROUP BY f.sk_cliente
    ),
    segmentado AS (
        SELECT
            sk_cliente,
            total_operaciones,
            CASE
                WHEN total_operaciones >= 20 THEN 'Alta'
                WHEN total_operaciones >= 5 THEN 'Media'
                ELSE 'Baja'
            END as segmento_actividad
        FROM actividad_cliente
    )
    SELECT
        segmento_actividad,
        COUNT(*) as cantidad_clientes,
        SUM(total_operaciones) as total_operaciones_segmento,
        ROUND(SUM(total_operaciones) / (SELECT SUM(total_operaciones) FROM segmentado) * 100, 1) as pct_del_volumen_total
    FROM segmentado
    GROUP BY segmento_actividad
    ORDER BY total_operaciones_segmento DESC
""").show(truncate=False)

+------------------+-----------------+--------------------------+---------------------+
|segmento_actividad|cantidad_clientes|total_operaciones_segmento|pct_del_volumen_total|
+------------------+-----------------+--------------------------+---------------------+
|Baja              |50328            |68126                     |77.9                 |
|Media             |1846             |14108                     |16.1                 |
|Alta              |125              |5268                      |6.0                  |
+------------------+-----------------+--------------------------+---------------------+

